# Level-2 Match-ups

In [1]:
!pip install -q point-collocation earthaccess

In [1]:
import earthaccess
earthaccess.__version__

'0.18.0'

In [2]:
auth = earthaccess.login()
# are we authenticated?
if not auth.authenticated:
    # ask for credentials and persist them in a .netrc file
    auth.login(strategy="interactive", persist=True)

In [16]:
import pandas as pd
df_points = pd.read_parquet('/home/jovyan/2026-go-bgc-hackweek/data/training_data.parquet')

In [32]:
%%time
import point_collocation as pc
plan = pc.plan(
    df_points[0:1],    
    data_source="earthaccess",
    source_kwargs={
        "short_name": "PACE_OCI_L2_AOP",
    },
    time_buffer="48h"
)

CPU times: user 18.7 ms, sys: 0 ns, total: 18.7 ms
Wall time: 525 ms


/opt/conda/lib/python3.13/site-packages/earthaccess/results.py:348: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  self["size"] = self.size()


In [33]:
plan.summary()

Plan: 1 points → 1 unique granule(s)
  Points with 0 matches : 0
  Points with >1 matches: 0
  Time buffer: 2 days 00:00:00

First 1 point(s):
  [0] lat=29.0826, lon=-17.2136, time=2024-03-05 01:16:05+00:00: 1 match(es)
    → https://obdaac-tea.earthdatacloud.nasa.gov/ob-cumulus-prod-public/PACE_OCI.20240305T131041.L2.OC_AOP.V3_2.nc


In [34]:
pace_l2 = {'xarray_open': 'dataset', 'merge': ['/geophysical_data', '/navigation_data']}
plan.open_dataset(0, open_method=pace_l2)

/opt/conda/lib/python3.13/site-packages/earthaccess/store.py:523: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  total_size = round(sum([granule.size() for granule in granules]) / 1024, 2)


open_method: {'xarray_open': 'dataset', 'merge': ['/geophysical_data', '/navigation_data'], 'open_kwargs': {'chunks': {}, 'engine': 'h5netcdf', 'decode_timedelta': False}, 'coords': 'auto', 'set_coords': True, 'dim_renames': None, 'auto_align_phony_dims': None, 'merge_kwargs': {}}


Geolocation auto detected with cf_xarray: ('longitude', 'latitude') — lon dims=('number_of_lines', 'pixels_per_line'), lat dims=('number_of_lines', 'pixels_per_line')
Points columns used: y='lat', x='lon', time='time'


<xarray.Dataset> Size: 3GB
Dimensions:     (wavelength: 172, number_of_lines: 1710, pixels_per_line: 1272)
Coordinates:
  * wavelength  (wavelength) float32 688B 346.0 348.5 350.9 ... 716.8 719.3
    longitude   (number_of_lines, pixels_per_line) float32 9MB dask.array<chunksize=(256, 1272), meta=np.ndarray>
    latitude    (number_of_lines, pixels_per_line) float32 9MB dask.array<chunksize=(256, 1272), meta=np.ndarray>
Dimensions without coordinates: number_of_lines, pixels_per_line
Data variables:
    band_index  (wavelength) float64 1kB dask.array<chunksize=(32,), meta=np.ndarray>
    Rrs         (number_of_lines, pixels_per_line, wavelength) float32 1GB dask.array<chunksize=(32, 256, 40), meta=np.ndarray>
    Rrs_unc     (number_of_lines, pixels_per_line, wavelength) float32 1GB dask.array<chunksize=(32, 256, 40), meta=np.ndarray>
    aot_865     (number_of_lines, pixels_per_line) float32 9MB dask.array<chunksize=(256, 1272), meta=np.ndarray>
    angstrom    (number_of_lines, pixels_per_line) float32 9MB dask.array<chunksize=(256, 1272), meta=np.ndarray>
    avw         (number_of_lines, pixels_per_line) float32 9MB dask.array<chunksize=(256, 1272), meta=np.ndarray>
    nflh        (number_of_lines, pixels_per_line) float32 9MB dask.array<chunksize=(256, 1272), meta=np.ndarray>
    l2_flags    (number_of_lines, pixels_per_line) int32 9MB dask.array<chunksize=(256, 1272), meta=np.ndarray>
    tilt        (number_of_lines) float32 7kB dask.array<chunksize=(32,), meta=np.ndarray>

In [26]:
%%time
# spatial_method="auto" defaults to "kdtree" in this case
res = pc.matchup(plan, variables=["Rrs"], open_method=pace_l2)

/opt/conda/lib/python3.13/site-packages/earthaccess/store.py:523: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  total_size = round(sum([granule.size() for granule in granules]) / 1024, 2)


CPU times: user 28.5 s, sys: 2.88 s, total: 31.4 s
Wall time: 1min 53s


In [28]:
120/10

12.0

In [29]:
res

,profile_id,time,lat,lon,pace_Rrs_346,pace_Rrs_348,pace_Rrs_351,pace_Rrs_353,pace_Rrs_356,pace_Rrs_358,...,Rrs_705,Rrs_706,Rrs_708,Rrs_709,Rrs_710,Rrs_711,Rrs_713,Rrs_714,Rrs_716,Rrs_719
0,4903485_0033,2024-03-05 01:16:05+00:00,29.082600,-17.213600,0.006416,0.006900,0.007244,0.007086,0.006898,0.007158,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,6904240_0068,2024-03-05 14:48:38.036000+00:00,59.971409,-31.955397,0.011284,0.010968,0.011102,0.010868,0.010602,0.010960,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,6904240_0068,2024-03-05 14:48:38.036000+00:00,59.971409,-31.955397,0.011284,0.010968,0.011102,0.010868,0.010602,0.010960,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,6904240_0068,2024-03-05 14:48:38.036000+00:00,59.971409,-31.955397,0.011284,0.010968,0.011102,0.010868,0.010602,0.010960,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,6904240_0068,2024-03-05 14:48:38.036000+00:00,59.971409,-31.955397,0.011284,0.010968,0.011102,0.010868,0.010602,0.010960,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,1902495_0002,2024-03-06 14:48:36.002000128+00:00,-19.579800,1.223800,0.009774,0.010156,0.010510,0.010670,0.010986,0.011558,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,1902495_0002,2024-03-06 14:48:36.002000128+00:00,-19.579800,1.223800,0.009774,0.010156,0.010510,0.010670,0.010986,0.011558,...,-0.000068,-0.000062,-0.000060,-0.000054,-0.000046,-0.000038,-0.000042,-0.000060,-0.000124,-0.000150
7,2903863_0040,2024-03-06 19:59:32.002000128+00:00,18.641800,-156.094300,0.009662,0.009998,0.010172,0.010452,0.010776,0.011138,...,0.000138,0.000140,0.000130,0.000122,0.000110,0.000102,0.000094,0.000088,0.000080,0.000070
8,2903863_0040,2024-03-06 19:59:32.002000128+00:00,18.641800,-156.094300,0.009662,0.009998,0.010172,0.010452,0.010776,0.011138,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,2902882_0079,2024-03-07 02:07:40+00:00,25.023887,148.098503,0.011946,0.012350,0.012650,0.012968,0.013314,0.013802,...,0.000190,0.000176,0.000168,0.000174,0.000162,0.000150,0.000148,0.000120,0.000120,0.000188
